In [1]:
# ============================================
# IMPORTS
# ============================================
import sys
import os
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

# Project imports
sys.path.append('..')
os.chdir('..')  
from src.data_pipeline.preprocess import *
from src.data_pipeline.features import *
from src.utils.eda_summary import *

# Visualization (optional for EDA within notebook)
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports loaded")



All imports loaded


In [2]:
# ============================================
# LOAD CONFIG
# ============================================
config_path = Path("configs/data_config.yaml")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract nested configs
dataset_cfg = config['dataset']['electronics']
paths_cfg = config['paths']
prep_cfg = config['preprocessing']

print("    Config loaded")
print(f"   Sample size: {dataset_cfg['sample_size']:,}")
print(f"   Chunk size:  {dataset_cfg['chunk_size']:,}")
print(f"   Item column: {prep_cfg['implicit_feedback']['item_col']}")


    Config loaded
   Sample size: 1,000,000
   Chunk size:  100,000
   Item column: parent_asin


##  Section 1: Load Raw Data


In [3]:
# ============================================
# LOAD RAW DATA
# ============================================
ratings_raw_path = Path(paths_cfg['raw_data']) / "Electronics_ratings.parquet"
metadata_raw_path = Path(paths_cfg['raw_data']) / "Electronics_metadata.parquet"

# Load
ratings_raw = pd.read_parquet(ratings_raw_path)
metadata_raw = pd.read_parquet(metadata_raw_path)

print(f"   Ratings loaded: {ratings_raw.shape}")
print(f"   Columns: {list(ratings_raw.columns)}")
print(f"   Memory: {ratings_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()
print(f"   Metadata loaded: {metadata_raw.shape}")
print(f"   Columns: {list(metadata_raw.columns)}")
print(f"   Memory: {metadata_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")


   Ratings loaded: (1000000, 10)
   Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
   Memory: 794.6 MB

   Metadata loaded: (1000000, 9)
   Columns: ['parent_asin', 'title', 'description', 'features', 'categories', 'price', 'store', 'average_rating', 'rating_number']
   Memory: 678.3 MB


In [4]:
# ============================================
# INITIAL EDA SNAPSHOT
# ============================================
print("=" * 60)
print("RATINGS — Initial State")
print("=" * 60)
print(ratings_raw.head(3))
print()
print("Missing values:")
print(summarize_missing_values(ratings_raw))
print()
print("Sparsity:")
print(calculate_sparsity(ratings_raw))

print("\n" + "=" * 60)
print("METADATA — Initial State")
print("=" * 60)
print(metadata_raw.head(3))
print()
print("Missing values:")
print(summarize_missing_values(metadata_raw))


RATINGS — Initial State
   rating                                    title  \
0     3.0        Smells like gasoline! Going back!   
1     1.0  Didn’t work at all lenses loose/broken.   
2     5.0                               Excellent!   

                                                text  \
0  First & most offensive: they reek of gasoline ...   
1  These didn’t work. Idk if they were damaged in...   
2  I love these. They even come with a carry case...   

                                              images        asin parent_asin  \
0  [{'attachment_type': 'IMAGE', 'large_image_url...  B083NRGZMM  B083NRGZMM   
1                                                 []  B07N69T6TM  B07N69T6TM   
2                                                 []  B01G8JO5F2  B01G8JO5F2   

                        user_id      timestamp  helpful_vote  \
0  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  1658185117948             0   
1  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  1592678549731             0   
2  AFKZENTNBQ7A7V7UXW

##  Section 2: Ratings Preprocessing


In [5]:
# ============================================
# 2.1 — BASIC CLEANING
# ============================================
df = ratings_raw.copy()

# Setup paths (used throughout Section 2)
processed_dir = Path(paths_cfg['processed_data'])
processed_dir.mkdir(parents=True, exist_ok=True)

# Step 1: Replace None-like strings (BEFORE dropping columns)
df = replace_none_strings(df, columns=prep_cfg['none_replacement']['columns'])

# Step 2: Remove missing values in title/text
df = remove_missing_values(df, subset=prep_cfg['missing_values']['subset'])

# Step 3: Remove duplicates
df = remove_duplicates(df, subset=prep_cfg['deduplication']['subset'])

# Step 4: Drop useless columns (AFTER None replacement)
df = drop_useless_columns(df, columns=prep_cfg['drop_columns'])

# Step 5: Convert timestamp
df = convert_timestamp(df, column='timestamp')

# Step 6: Convert to integer
df = convert_to_integer(df, columns=prep_cfg['convert_to_integer']['columns'])

print(f"   After basic cleaning: {df.shape}")
print(f"   Columns: {list(df.columns)}")
print(f"   Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")


[WARNING] Columns not found, skipping: ['description', 'features', 'categories', 'store']
[INFO] Replaced None strings in 'title' (string column)
[INFO] Replaced None strings in 'text' (string column)
[WARNING] Could not process 'images': The truth value of an empty array is ambiguous. Use `array.size > 0` to check that an array is not empty.
[INFO] Removed 374 rows with missing values | Remaining: 999626
[INFO] Removed 2495 duplicate rows | Remaining: 997131
[INFO] Dropped columns: ['images']
[INFO] Converted 'timestamp' to datetime format
[INFO] Converted 'verified_purchase' to integer type
   After basic cleaning: (997131, 9)
   Columns: ['rating', 'title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
   Date range: 1998-03-03 12:17:02 → 2023-03-20 20:48:25.319000


In [6]:
# ============================================
# 2.2 — OUTLIER HANDLING (Log1p, NOT IQR)
# ============================================
# Reason: Preserves Long Tail signal
# IQR capping would destroy the natural distribution of helpful_vote
#
# NOTE: handle_outliers() OVERWRITES the original column
#       helpful_vote is now log1p(helpful_vote), not the raw count


df = handle_outliers(df, columns=prep_cfg['outliers']['columns'])

print("helpful_vote — Distribution after Log1p:")
print(summarize_distribution(df, 'helpful_vote'))  
print()
print("Original column (helpful_vote) preserved for reference.")
print("New column (helpful_vote_log1p) will be used downstream.")


[INFO] Applied log1p transformation to 'helpful_vote'
helpful_vote — Distribution after Log1p:
{'column': 'helpful_vote', 'missing_count': 0, 'missing_percentage': 0.0, 'count': 997131, 'mean': np.float64(0.2865), 'median': np.float64(0.0), 'std': np.float64(0.6465), 'min': np.float64(0.0), 'q25': np.float64(0.0), 'q75': np.float64(0.0), 'max': np.float64(9.0737), 'skewness': np.float64(3.197), 'kurtosis': np.float64(13.6836)}

Original column (helpful_vote) preserved for reference.
New column (helpful_vote_log1p) will be used downstream.


In [7]:
# ============================================
# 2.3 — SPAM DETECTION
# ============================================
# NOTE: detect_spam() REMOVES spam users (not just flags them)
# Rule: Users with >5 reviews/day OR <10s between consecutive reviews
# Result: All reviews from flagged users are dropped

spam_cfg = prep_cfg['spam_detection']
rows_before = len(df)
df = detect_spam(
    df,
    max_reviews_per_day=spam_cfg['max_reviews_per_day'],
    min_time_gap=spam_cfg['min_time_gap']
)
rows_after = len(df)

print(f"   Spam users removed:")
print(f"   Before: {rows_before:,} rows")
print(f"   After:  {rows_after:,} rows")
print(f"   Removed: {rows_before - rows_after:,} rows from spam users")


[INFO] Spam detection: Removed 7686 spam users
[INFO] Rows: 997131 -> 799158
   Spam users removed:
   Before: 997,131 rows
   After:  799,158 rows
   Removed: 197,973 rows from spam users


In [8]:
# ============================================
# 2.4 — TEXT FILTERING
# ============================================
text_cfg = prep_cfg['text_filter']
rows_before = len(df)
df = filter_text(
    df,
    column='text',
    min_words=text_cfg['min_words'],
    max_words=text_cfg['max_words']
)
rows_after = len(df)

print(f"   Text filter applied:")
print(f"   Before: {rows_before:,}")
print(f"   After:  {rows_after:,}")
print(f"   Removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")


[INFO] Text filtering: 719977 valid texts for NLP processing
[INFO] Marked 79181 short texts as None (< 5 words)
[INFO] Truncated 26563 long texts to 250 words
   Text filter applied:
   Before: 799,158
   After:  799,158
   Removed: 0 (0.00%)


In [9]:
# ============================================
# 2.5 —   DO NOT APPLY filter_high_signal_ratings
# ============================================
# Reason: ALS uses IMPLICIT feedback
#         All interactions (1-5 stars) = positive signal
#         Removing low ratings would bias the model

#  DO NOT UNCOMMENT — kept here as documentation
# df = filter_high_signal_ratings(df, min_rating=4.0)

print("   SKIPPED: filter_high_signal_ratings")
print("   Reason: ALS treats all interactions as implicit positive feedback")
print("   Removing low ratings would introduce selection bias")


   SKIPPED: filter_high_signal_ratings
   Reason: ALS treats all interactions as implicit positive feedback
   Removing low ratings would introduce selection bias


In [10]:
# ============================================
# 2.6 — DEDUPLICATE USER-ITEM PAIRS
# ============================================
# Rule: If user has multiple reviews for same item, keep the latest
rows_before = len(df)
df = deduplicate_user_item(
    df,
    user_col=prep_cfg['implicit_feedback']['user_col'],
    item_col=prep_cfg['implicit_feedback']['item_col']
)
rows_after = len(df)

print(f"   Deduplication applied:")
print(f"   Before: {rows_before:,}")
print(f"   After:  {rows_after:,}")
print(f"   Removed: {rows_before - rows_after:,} duplicate user-item pairs")


[INFO] Deduplication ('user_id' x 'parent_asin'): 1077 duplicates removed
[INFO] Rows: 799158 -> 798081
   Deduplication applied:
   Before: 799,158
   After:  798,081
   Removed: 1,077 duplicate user-item pairs


In [11]:
# ============================================
# 2.7 — ITERATIVE K-CORE FILTERING (10/10)
# ============================================
# CRITICAL for ALS: Removes users/items with <10 interactions
# Applied iteratively until stable (max 3 iterations)
# Reason: Sparse users/items add noise and hurt model quality

kcore_cfg = prep_cfg['iterative_k_core']
rows_before = len(df)
df = apply_iterative_k_core(
    df,
    columns_config=kcore_cfg['columns_config'],
    max_iterations=kcore_cfg['max_iterations']
)
rows_after = len(df)

print(f"   Iterative k-core (10/10) applied:")
print(f"   Before: {rows_before:,}")
print(f"   After:  {rows_after:,}")
print(f"   Removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")
print()
print("New stats:")
print(f"   Unique users: {df['user_id'].nunique():,}")
print(f"   Unique items: {df['parent_asin'].nunique():,}")
print(f"   Sparsity: {calculate_sparsity(df)}")



[INFO] Starting iterative K-Core filtering (max 3 iterations)
    [Iter 1] user_id (k>3): 113144 removed | 64345 remaining
    [Iter 1] parent_asin (k>3): 172424 removed | 29644 remaining
    [Iter 2] user_id (k>3): 22300 removed | 41471 remaining
    [Iter 2] parent_asin (k>3): 4286 removed | 25346 remaining
    [Iter 3] user_id (k>3): 1830 removed | 39641 remaining
    [Iter 3] parent_asin (k>3): 468 removed | 24878 remaining
[WARNING] K-Core did not converge after 3 iterations
[INFO] Final K-Core result: 306159 rows remaining

   Iterative k-core (10/10) applied:
   Before: 798,081
   After:  306,159
   Removed: 491,922 (61.64%)

New stats:
   Unique users: 39,641
   Unique items: 24,878
   Sparsity: {'n_users': 39641, 'n_items': 24878, 'n_interactions': 306159, 'possible_interactions': 986188798, 'sparsity_percentage': 99.97, 'density_percentage': 0.03}


In [12]:
# ============================================
# 2.8 — TOP-N ITEMS FILTER (10,000 items)
# ============================================
# Reason: Reduces computational cost + removes extreme long tail
# 10K items is a sweet spot for 1M dataset (balance coverage vs speed)

topn_cfg = prep_cfg['filter_top_n']
rows_before = len(df)
df = filter_top_n_items(
    df,
    item_col=topn_cfg['item_col'],
    top_n=topn_cfg['top_n']
)
rows_after = len(df)

print(f"   Top-{topn_cfg['top_n']} items filter applied:")
print(f"   Before: {rows_before:,}")
print(f"   After:  {rows_after:,}")
print(f"   Removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")


[INFO] Top-N item filtering (top 10000): 14878 items removed
[INFO] Rows: 306159 -> 232441
   Top-10000 items filter applied:
   Before: 306,159
   After:  232,441
   Removed: 73,718 (24.08%)


In [13]:
# ============================================
# 2.9 — CONFIDENCE WEIGHTS + IMPLICIT FEEDBACK
# ============================================
weight_cfg = prep_cfg['review_weight']

# Step 1: Apply confidence weight (verified vs unverified)
# Function signature: apply_confidence_weight(df, verified_col, weight_col, verified_weight, unverified_weight)
df = apply_confidence_weight(
    df,
    verified_col='verified_purchase',
    weight_col='confidence_weight',
    verified_weight=weight_cfg['verified'],      
    unverified_weight=weight_cfg['unverified']   
)

# Step 2: Set implicit feedback weight (interaction exists = 1.0)
df = set_implicit_feedback_weight(
    df,
    weight_col='implicit_weight',
    interaction_weight=1.0
)

# Step 3: Combined weight (confidence × implicit)
df['final_weight'] = df['confidence_weight'] * df['implicit_weight']

print(" Weights added:")
print(df[['verified_purchase', 'confidence_weight', 'implicit_weight', 'final_weight']].head())
print()
print("Weight distribution:")
print(df['final_weight'].value_counts())


[INFO] Applied confidence weighting:
       Verified purchases: weight = 1.0
       Unverified purchases: weight = 0.7
[INFO] Set implicit feedback weight: all interactions = 1.0
 Weights added:
        verified_purchase  confidence_weight  implicit_weight  final_weight
524458                  1                1.0              1.0           1.0
524457                  1                1.0              1.0           1.0
524456                  1                1.0              1.0           1.0
524454                  1                1.0              1.0           1.0
85786                   1                1.0              1.0           1.0

Weight distribution:
final_weight
1.0    216049
0.7     16392
Name: count, dtype: int64


In [14]:
# ============================================
# 2.10 — ENCODE LABELS (user_id, parent_asin → int)
# ============================================
# CRITICAL: Use parent_asin (NOT asin) — consistent with train.py
# Function returns: (df, encoders_dict)

# BACKUP original parent_asin before encoding
# Reason: We need it later to match with metadata (which has string ASINs)
df['parent_asin_original'] = df['parent_asin'].copy()

encode_cols = prep_cfg['encode']['columns']
df, encoders = encode_labels(df, columns=encode_cols, encoders=None)

# Save encoders for later use (in train.py, evaluate.py, RAG, etc.)
import joblib
encoders_path = Path(paths_cfg['processed_data']) / "encoders.pkl"
encoders_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(encoders, encoders_path)

print(f"   Labels encoded: {encode_cols}")
print(f"   user_id: {df['user_id'].nunique():,} unique")
print(f"   parent_asin: {df['parent_asin'].nunique():,} unique")
print(f"   Encoders saved to: {encoders_path}")
print(f"   parent_asin_original saved (for metadata matching)")


[INFO] Encoded 'user_id' (fitted) -> 39590 unique labels
[INFO] Encoded 'parent_asin' (fitted) -> 10000 unique labels
   Labels encoded: ['user_id', 'parent_asin']
   user_id: 39,590 unique
   parent_asin: 10,000 unique
   Encoders saved to: data\processed\encoders.pkl
   parent_asin_original saved (for metadata matching)


In [15]:
# ============================================
# 2.11 — ADD USER SEGMENT (Light/Medium/Heavy)
# ============================================
df = add_user_segment(df, light_max=2, medium_max=10)

print("  User segment distribution:")
print(df['user_segment'].value_counts())
print()
print("Percentages:")
print(df['user_segment'].value_counts(normalize=True) * 100)


[INFO] User segments:
user_segment
Medium    162360
Heavy      63788
Light       6293
Name: count, dtype: int64
  User segment distribution:
user_segment
Medium    162360
Heavy      63788
Light       6293
Name: count, dtype: int64

Percentages:
user_segment
Medium    69.849983
Heavy     27.442663
Light      2.707354
Name: proportion, dtype: float64


In [16]:
# ============================================
# 2.12 — SANITY CHECK (After All Preprocessing)
# ============================================
print("=" * 60)
print("RATINGS — After All Preprocessing Steps")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()

# Check that all expected normalized columns exist
expected_norm_cols = ['helpful_vote_norm', 'confidence_norm', 'final_weight_norm']
# (These are in train_df/val_df/test_df, not df, so this is a reminder)
print("Normalized features (in train/val/test):")
for col in expected_norm_cols:
    print(f"   ✓ {col}")

print()
print("Missing values:")
print(summarize_missing_values(df))
print()
print("Sparsity:")
print(calculate_sparsity(df))
print()
print("Date range:")
print(f"   Min: {df['timestamp'].min()}")
print(f"   Max: {df['timestamp'].max()}")
print()
print("Sample rows:")
print(df.head(3))


RATINGS — After All Preprocessing Steps
Shape: (232441, 14)
Columns: ['rating', 'title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'confidence_weight', 'implicit_weight', 'final_weight', 'parent_asin_original', 'user_segment']

Normalized features (in train/val/test):
   ✓ helpful_vote_norm
   ✓ confidence_norm
   ✓ final_weight_norm

Missing values:
{'total_rows': 232441, 'columns_missing': {'text': {'count': 22373, 'percentage': np.float64(9.63)}}, 'columns_complete': ['rating', 'title', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'confidence_weight', 'implicit_weight', 'final_weight', 'parent_asin_original', 'user_segment'], 'overall_completeness': 99.31}

Sparsity:
{'n_users': 39590, 'n_items': 10000, 'n_interactions': 232441, 'possible_interactions': 395900000, 'sparsity_percentage': 99.94, 'density_percentage': 0.06}

Date range:
   Min: 2000-07-12 19:54:59
   Max: 2023-03-19 17:13:02.067000

##  Section 3: Metadata Preprocessing


In [17]:
# ============================================
# 3.1 — METADATA BASIC CLEANING
# ============================================
df_meta = metadata_raw.copy()

# Step 1: Replace None-like strings (BEFORE dropping columns)
df_meta = replace_none_strings(
    df_meta, 
    columns=prep_cfg['none_replacement']['columns']
)

# Step 2: Remove missing parent_asin (critical for joining)
df_meta = remove_missing_values(df_meta, subset=['parent_asin'])

# Step 3: Remove duplicates
df_meta = remove_duplicates(df_meta, subset=['parent_asin'])

print(f"   Metadata cleaned: {df_meta.shape}")
print(f"   Columns: {list(df_meta.columns)}")


[WARNING] Columns not found, skipping: ['text', 'images']
[INFO] Replaced None strings in 'title' (string column)
[WARNING] Could not process 'description': The truth value of an empty array is ambiguous. Use `array.size > 0` to check that an array is not empty.
[WARNING] Could not process 'features': The truth value of an empty array is ambiguous. Use `array.size > 0` to check that an array is not empty.
[WARNING] Could not process 'categories': The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
[INFO] Replaced None strings in 'store' (string column)
[INFO] Removed 0 rows with missing values | Remaining: 1000000
[INFO] Removed 0 duplicate rows | Remaining: 1000000
   Metadata cleaned: (1000000, 9)
   Columns: ['parent_asin', 'title', 'description', 'features', 'categories', 'price', 'store', 'average_rating', 'rating_number']


In [18]:
# ============================================
# 3.2 — ADD STORE TIER
# ============================================
# Premium: large_min (50+) products + rating >= 4.4
# Trusted: product_count >= small_max (2) + rating >= rating_threshold
# Small Seller: everything else

df_meta = add_store_tier(
    df_meta,
    small_max=2,
    large_min=50,
    rating_threshold=4.4
)

print("Store tier distribution:")
print(df_meta['store_tier'].value_counts())
print()
print("Percentages:")
print(df_meta['store_tier'].value_counts(normalize=True) * 100)


[INFO] Store tier distribution:
store_tier
Trusted         926183
Premium          44687
Small Seller     23400
Name: count, dtype: int64
Store tier distribution:
store_tier
Trusted         926183
Premium          44687
Small Seller     23400
Name: count, dtype: int64

Percentages:
store_tier
Trusted         93.152061
Premium          4.494453
Small Seller     2.353485
Name: proportion, dtype: float64


In [19]:
# ============================================
# 3.3 — FILTER METADATA TO ITEMS IN RATINGS
# ============================================
# Reason: We only need metadata for items that have interactions
# Reduces memory + speeds up RAG later

# Use parent_asin_original (string) for matching with metadata
train_items = set(df['parent_asin_original'].unique())  #  Fixed: use original
df_meta = df_meta[df_meta['parent_asin'].isin(train_items)].reset_index(drop=True)

print(f"   Metadata filtered to {len(train_items):,} train items")
print(f"   Metadata rows: {len(df_meta):,}")
print(f"   Coverage: {len(df_meta) / len(train_items) * 100:.2f}%")


   Metadata filtered to 10,000 train items
   Metadata rows: 8,401
   Coverage: 84.01%


In [20]:
# ============================================
# 3.4 — HANDLE MISSING VALUES IN METADATA
# ============================================
# Strategy: Keep as-is (for RAG and content-based)
# - price: 65% missing → kept as NaN (optional feature)
# - features: 25.5% empty → use description as fallback later
# - description: critical for RAG → drop if missing

# Debug: Check shape before filtering
print(f"Before filter: df_meta.shape = {df_meta.shape}")
print(f"   description missing: {df_meta['description'].isna().sum()}")
print()

rows_before = len(df_meta)
df_meta = remove_missing_values(df_meta, subset=['description'])
rows_after = len(df_meta)

print(f"\n Missing 'description' handled:")
print(f"   Before: {rows_before:,}")
print(f"   After:  {rows_after:,}")

# Defensive: If empty, warn user
if len(df_meta) == 0:
    print("   WARNING: df_meta is EMPTY after filtering!")
    print("   This means ALL items in train_df have missing descriptions")
    print("   Possible solutions:")
    print("   1. Use a different 'required' column (e.g., title)")
    print("   2. Use a fallback strategy (empty description → '')")
    print("   3. Re-evaluate the metadata preprocessing")

# Verify final missing state
print()
print("Final missing values:")
print(summarize_missing_values(df_meta))


Before filter: df_meta.shape = (8401, 10)
   description missing: 0

[INFO] Removed 0 rows with missing values | Remaining: 8401

 Missing 'description' handled:
   Before: 8,401
   After:  8,401

Final missing values:
{'total_rows': 8401, 'columns_missing': {'price': {'count': 2436, 'percentage': np.float64(29.0)}, 'store': {'count': 4, 'percentage': np.float64(0.05)}, 'store_tier': {'count': 4, 'percentage': np.float64(0.05)}}, 'columns_complete': ['parent_asin', 'title', 'description', 'features', 'categories', 'average_rating', 'rating_number'], 'overall_completeness': 97.09}


In [21]:
# ============================================
# 3.5 — SAVE METADATA
# ============================================
metadata_out_path = Path(paths_cfg['processed_data']) / paths_cfg['metadata_file']
metadata_out_path.parent.mkdir(parents=True, exist_ok=True)

df_meta.to_parquet(metadata_out_path, index=False)

print(f"   Metadata saved: {metadata_out_path}")
print(f"   Size: {metadata_out_path.stat().st_size / 1e6:.1f} MB")
print(f"   Rows: {len(df_meta):,}")


   Metadata saved: data\processed\metadata_processed.parquet
   Size: 8.8 MB
   Rows: 8,401


##  Section 4: Time-Based Split


In [22]:
# ============================================
# 4.1 — TIME-BASED SPLIT
# ============================================
# Rule:
#   - Train: timestamp.year < 2021
#   - Val:   timestamp.year >= 2021 and < 2022
#   - Test:  timestamp.year >= 2022  (SEALED)
#
# Reason: Prevents data leakage + simulates real production

split_cfg = prep_cfg['split']
train_df, val_df, test_df = time_based_split(  #  Fixed: returns tuple
    df,
    column='timestamp',
    val_year=split_cfg['val_year'],
    test_year=split_cfg['test_year']
)

print("=" * 60)
print("TIME-BASED SPLIT RESULTS")
print("=" * 60)
print(f"Train: {len(train_df):,} rows ({len(train_df)/len(df)*100:.1f}%)")
print(f"   Date range: {train_df['timestamp'].min()} → {train_df['timestamp'].max()}")
print()
print(f"Val:   {len(val_df):,} rows ({len(val_df)/len(df)*100:.1f}%)")
print(f"   Date range: {val_df['timestamp'].min()} → {val_df['timestamp'].max()}")
print()
print(f"Test:  {len(test_df):,} rows ({len(test_df)/len(df)*100:.1f}%)")
print(f"   Date range: {test_df['timestamp'].min()} → {test_df['timestamp'].max()}")


[INFO] Time-based split:
       Train: 188679 rows (81.2%)
       Val:   23040 rows (9.9%)
       Test:  20722 rows (8.9%)
TIME-BASED SPLIT RESULTS
Train: 188,679 rows (81.2%)
   Date range: 2000-07-12 19:54:59 → 2020-12-31 23:55:33.346000

Val:   23,040 rows (9.9%)
   Date range: 2021-01-01 00:03:57.161000 → 2021-12-31 23:43:53.030000

Test:  20,722 rows (8.9%)
   Date range: 2022-01-01 00:21:50.647000 → 2023-03-19 17:13:02.067000


In [23]:
# ============================================
# 4.2 — STATS COMPARISON
# ============================================
comparison = pd.DataFrame({
    'Split': ['Train', 'Val', 'Test'],
    'Rows': [len(train_df), len(val_df), len(test_df)],
    'Users': [train_df['user_id'].nunique(), val_df['user_id'].nunique(), test_df['user_id'].nunique()],
    'Items': [train_df['parent_asin'].nunique(), val_df['parent_asin'].nunique(), test_df['parent_asin'].nunique()],
    'Sparsity': [
        calculate_sparsity(train_df),
        calculate_sparsity(val_df),
        calculate_sparsity(test_df)
    ]
})
print(comparison)


   Split    Rows  Users  Items  \
0  Train  188679  38128   9504   
1    Val   23040  13730   4849   
2   Test   20722  12173   4226   

                                            Sparsity  
0  {'n_users': 38128, 'n_items': 9504, 'n_interac...  
1  {'n_users': 13730, 'n_items': 4849, 'n_interac...  
2  {'n_users': 12173, 'n_items': 4226, 'n_interac...  


In [24]:
# ============================================
# 4.3 — NORMALIZE NUMERICAL FEATURES
# ============================================
# Reason: Prepare for hybrid/neural models that need normalized features
# - Current CF models (ALS/BPR/KNN/SVD) don't strictly require this
# - But helpful_vote and confidence_weight are useful side features
# - Saves scaler for production inference consistency
#
# Source columns (from earlier cells):
#   - helpful_vote:       from 2.2 (handle_outliers OVERWRITES with log1p)
#   - confidence_weight:  from 2.9 (apply_confidence_weight)
#   - final_weight:       from 2.9 (confidence × implicit)
#
#  IMPORTANT: helpful_vote now contains log1p values, NOT raw counts
#              We normalize the LOG values (not the original counts)
#
#  CRITICAL: Must be done AFTER time_based_split (4.1)
#              Train-only fit, val/test transform (no data leakage)

from sklearn.preprocessing import MinMaxScaler
import joblib

# -------- Feature 1: helpful_vote (log1p values) --------
scaler_helpful = MinMaxScaler()
train_df['helpful_vote_norm'] = scaler_helpful.fit_transform(
    train_df[['helpful_vote']]      # log1p values
)
val_df['helpful_vote_norm'] = scaler_helpful.transform(
    val_df[['helpful_vote']]
)
test_df['helpful_vote_norm'] = scaler_helpful.transform(
    test_df[['helpful_vote']]
)

# -------- Feature 2: confidence_weight --------
scaler_conf = MinMaxScaler()
train_df['confidence_norm'] = scaler_conf.fit_transform(
    train_df[['confidence_weight']]
)
val_df['confidence_norm'] = scaler_conf.transform(
    val_df[['confidence_weight']]
)
test_df['confidence_norm'] = scaler_conf.transform(
    test_df[['confidence_weight']]
)

# -------- Feature 3: final_weight --------
scaler_final = MinMaxScaler()
train_df['final_weight_norm'] = scaler_final.fit_transform(
    train_df[['final_weight']]
)
val_df['final_weight_norm'] = scaler_final.transform(
    val_df[['final_weight']]
)
test_df['final_weight_norm'] = scaler_final.transform(
    test_df[['final_weight']]
)

# -------- Save all scalers --------
scalers = {
    'helpful': scaler_helpful,
    'confidence': scaler_conf,
    'final_weight': scaler_final
}
scalers_path = processed_dir / "scalers.pkl"
joblib.dump(scalers, scalers_path)

print("    Numerical features normalized (post-split, no data leakage):")
print(f"   helpful_vote_norm:  {train_df['helpful_vote_norm'].min():.3f} → {train_df['helpful_vote_norm'].max():.3f}")
print(f"   confidence_norm:    {train_df['confidence_norm'].min():.3f} → {train_df['confidence_norm'].max():.3f}")
print(f"   final_weight_norm:  {train_df['final_weight_norm'].min():.3f} → {train_df['final_weight_norm'].max():.3f}")
print()
print(f"   Scalers saved: {scalers_path}")


    Numerical features normalized (post-split, no data leakage):
   helpful_vote_norm:  0.000 → 1.000
   confidence_norm:    0.000 → 1.000
   final_weight_norm:  0.000 → 1.000

   Scalers saved: data\processed\scalers.pkl


In [25]:
# ============================================
# 4.4 — ADD FEATURES (CRITICAL: Train-Only)
# ============================================
#  user_verified_ratio and item_avg_rating must be computed on TRAIN only
# Then MAPPED to val and test (prevent data leakage)

# Compute features on train
train_df = add_features(train_df)

# Map features to val (using train's statistics)
user_verified_map = train_df.groupby('user_id')['verified_purchase'].mean().to_dict()
item_avg_rating_map = train_df.groupby('parent_asin')['rating'].mean().to_dict()
user_avg_helpful_map = train_df.groupby('user_id')['helpful_vote'].mean().to_dict()

# Apply to val
val_df['user_verified_ratio'] = val_df['user_id'].map(user_verified_map).fillna(0.5)
val_df['item_avg_rating'] = val_df['parent_asin'].map(item_avg_rating_map).fillna(train_df['rating'].mean())
val_df['is_weekend'] = val_df['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)

# Apply to test (still sealed, but we create the columns for consistency)
test_df['user_verified_ratio'] = test_df['user_id'].map(user_verified_map).fillna(0.5)
test_df['item_avg_rating'] = test_df['parent_asin'].map(item_avg_rating_map).fillna(train_df['rating'].mean())
test_df['is_weekend'] = test_df['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)

print("Features added (train-computed, val/test mapped)")
print()
print("Train feature stats:")
print(train_df[['user_verified_ratio', 'item_avg_rating', 'is_weekend']].describe())


[INFO] Added features: user_verified_ratio, item_avg_rating, is_weekend
Features added (train-computed, val/test mapped)

Train feature stats:
       user_verified_ratio  item_avg_rating     is_weekend
count        188679.000000    188679.000000  188679.000000
mean              0.924735         4.285501       0.258163
std               0.153285         0.495070       0.437625
min               0.000000         1.000000       0.000000
25%               0.900000         4.000000       0.000000
50%               1.000000         4.363636       0.000000
75%               1.000000         4.619048       1.000000
max               1.000000         5.000000       1.000000


##  Section 5: Save & Final Validation


In [26]:
# ============================================
# 5.1 — SAVE TRAIN/VAL/TEST
# ============================================
processed_dir = Path(paths_cfg['processed_data'])
processed_dir.mkdir(parents=True, exist_ok=True)

# Define paths
train_path = processed_dir / paths_cfg['train_file']
val_path   = processed_dir / paths_cfg['val_file']
test_path  = processed_dir / paths_cfg['test_file']

#  CRITICAL: Save with correct names (NO SWAP!)
train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)
test_df.to_parquet(test_path, index=False)

print("✅ Files saved:")
for name, path in [('Train', train_path), ('Val', val_path), ('Test', test_path)]:
    size_mb = path.stat().st_size / 1e6
    print(f"   {name:5s}: {path} ({size_mb:.1f} MB, {len(pd.read_parquet(path)):,} rows)")


✅ Files saved:
   Train: data\processed\train.parquet (40.6 MB, 188,679 rows)
   Val  : data\processed\val.parquet (4.7 MB, 23,040 rows)
   Test : data\processed\test.parquet (4.4 MB, 20,722 rows)


In [27]:
# ============================================
# 5.2 — FINAL VALIDATION (Reload from disk)
# ============================================
# Goal: Make sure files are readable + schemas are consistent

print("=" * 60)
print("FINAL VALIDATION — Reload from Disk")
print("=" * 60)

train_check = pd.read_parquet(train_path)
val_check   = pd.read_parquet(val_path)
test_check  = pd.read_parquet(test_path)

# Check shapes
assert len(train_check) == len(train_df), " Train shape mismatch!"
assert len(val_check) == len(val_df), " Val shape mismatch!"
assert len(test_check) == len(test_df), " Test shape mismatch!"
print("Shapes match in-memory vs on-disk")

# Check schemas
assert list(train_check.columns) == list(train_df.columns), " Train schema mismatch!"
assert list(val_check.columns) == list(val_df.columns), " Val schema mismatch!"
assert list(test_check.columns) == list(test_df.columns), " Test schema mismatch!"
print("Schemas match in-memory vs on-disk")

# Check dtypes
assert train_check.dtypes.equals(train_df.dtypes), " Train dtypes mismatch!"
print("Dtypes preserved (Parquet working correctly)")
# Check Normalizatin 
expected_norm_cols = ['helpful_vote_norm', 'confidence_norm', 'final_weight_norm']
for col in expected_norm_cols:
    assert col in train_check.columns, f" Missing normalized column: {col}"
    assert train_check[col].min() >= 0.0, f" {col} has negative values"
    assert train_check[col].max() <= 1.0, f" {col} exceeds 1.0"
print(" Normalized features validated (range [0, 1])")

print()
print("=" * 60)
print("ALL CHECKS PASSED — Preprocessing Complete")
print("=" * 60)


FINAL VALIDATION — Reload from Disk
Shapes match in-memory vs on-disk
Schemas match in-memory vs on-disk
Dtypes preserved (Parquet working correctly)
 Normalized features validated (range [0, 1])

ALL CHECKS PASSED — Preprocessing Complete
